# 01_EDA

Exploratory Data Analysis for the India credit card transactions dataset.

This notebook reproduces the same visual outputs generated by `src/patterns.py` and saves them into `notebooks/figures/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", None)

DATA_PATH = Path("../data/raw/Credit card transactions - India - Simple.csv")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(DATA_PATH).rename(
    columns={
        "City": "city",
        "Date": "date",
        "Card Type": "card_type",
        "Exp Type": "exp_type",
        "Gender": "gender",
        "Amount": "amount",
    }
)

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"])

print(df.shape)
df.head()

In [ ]:
# 1) amount_by_category.png
by_category = df.groupby("exp_type", as_index=False)["amount"].sum().sort_values("amount", ascending=False)
plt.figure(figsize=(10, 5))
plt.bar(by_category["exp_type"], by_category["amount"])
plt.xticks(rotation=45, ha="right")
plt.title("Total Amount by Category")
plt.tight_layout()
plt.savefig(FIG_DIR / "amount_by_category.png", dpi=150)
plt.close()

# 2) amount_distribution.png
plt.figure(figsize=(8, 5))
plt.hist(df["amount"], bins=40)
plt.title("Amount Distribution")
plt.xlabel("Amount")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(FIG_DIR / "amount_distribution.png", dpi=150)
plt.close()

# 3) category_proportions.png
plt.figure(figsize=(7, 7))
plt.pie(by_category["amount"], labels=by_category["exp_type"], autopct="%1.1f%%", startangle=140)
plt.title("Category Proportions")
plt.tight_layout()
plt.savefig(FIG_DIR / "category_proportions.png", dpi=150)
plt.close()

# 4) correlation_heatmap.png
corr_df = df.copy()
corr_df["month"] = corr_df["date"].dt.month
corr_df["day"] = corr_df["date"].dt.day
corr_df["weekday"] = corr_df["date"].dt.dayofweek
cols = ["amount", "month", "day", "weekday"]
corr = corr_df[cols].corr()
plt.figure(figsize=(6, 5))
plt.imshow(corr, cmap="coolwarm", interpolation="nearest")
plt.colorbar()
plt.xticks(range(len(cols)), cols, rotation=45, ha="right")
plt.yticks(range(len(cols)), cols)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig(FIG_DIR / "correlation_heatmap.png", dpi=150)
plt.close()

# 5) expense_type_distribution.png
exp_counts = df["exp_type"].value_counts()
plt.figure(figsize=(10, 5))
plt.bar(exp_counts.index, exp_counts.values)
plt.xticks(rotation=45, ha="right")
plt.title("Expense Type Distribution")
plt.tight_layout()
plt.savefig(FIG_DIR / "expense_type_distribution.png", dpi=150)
plt.close()

# 6) monthly_trend.png
monthly = (
    df.assign(month=df["date"].dt.to_period("M").dt.to_timestamp())
    .groupby("month", as_index=False)["amount"]
    .sum()
    .sort_values("month")
)
plt.figure(figsize=(10, 5))
plt.plot(monthly["month"], monthly["amount"], marker="o")
plt.title("Monthly Trend")
plt.xlabel("Month")
plt.ylabel("Total Amount")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "monthly_trend.png", dpi=150)
plt.close()

# 7) top_cities.png
top_cities = df.groupby("city", as_index=False)["amount"].sum().sort_values("amount", ascending=False).head(10)
plt.figure(figsize=(10, 5))
plt.bar(top_cities["city"], top_cities["amount"])
plt.xticks(rotation=45, ha="right")
plt.title("Top Cities by Amount")
plt.tight_layout()
plt.savefig(FIG_DIR / "top_cities.png", dpi=150)
plt.close()

print("All figure files saved in notebooks/figures")